In [ ]:
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
os.listdir(path)

In [ ]:
os.listdir(path + '/dataset')

In [ ]:
os.listdir(path + '/dataset' + '/images')

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
class customData():
  def __init__(self, image_paths, mask_paths, transform=None, mask_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.mask_transform = mask_transform

  def __len__(self):
        return len(self.image_paths)

  def __getitem__(self, idx):
      # 1. Load Image & Mask
      image = Image.open(self.image_paths[idx]).convert("RGB")
      mask = Image.open(self.mask_paths[idx]).convert("L") # Keep as L (grayscale)

      # 2. Apply Transform
      if self.transform:
          image = self.transform(image)
      if self.mask_transform:
           mask = self.mask_transform(mask)

      mask = remap_mask(mask)

      return image, mask

In [ ]:
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import DataLoader

# Define Paths
root_dir = os.path.join(path, "dataset")
# Extract all the images and sort them
all_images = sorted(glob.glob(f"{root_dir}/images/*.jpg"))
all_masks  = sorted(glob.glob(f"{root_dir}/masks/*.png"))

# Split Data
train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

# Define Transform
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()         # ToTensor does two things: Convert to tensor + scaling (divide by 255)
])
transform_mask = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.PILToTensor()       # PILToTensor does one thing: Convert to tensor only (mask should not be scaled!!)
])

# Create Datasets
train_dataset = customData(train_imgs, train_masks, transform=transform, mask_transform=transform_mask)
test_dataset  = customData(test_imgs,  test_masks,  transform=transform, mask_transform=transform_mask)

# Defind the loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# Check Output
img, mask = train_dataset[0]
print(f"Img Shape: {img.shape}")   # [3, 256, 256]
print(f"Mask Shape: {mask.shape}") # [1, 256, 256]
print(f"Unique Classes: {torch.unique(mask)}")

In [ ]:
# Display 4 images with their masks
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    image, mask = train_dataset[i]

    # Display image
    axes[0, i].imshow(image.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"Orgainl Image {i+1}")
    axes[0, i].axis("off")

    # Display mask
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title(f"Tumor Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Defind the model with efficientnet-b1 pretraind model
model = smp.Unet(
  encoder_name="efficientnet-b1",
  encoder_weights="imagenet",
  in_channels=3,
  classes=1,
).to(device)

model = model.to(device)

In [ ]:
# TO DO
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device
    images, masks = images.to(device), masks.to(device).float()

    outputs = model(images)
    loss = criterion(outputs, masks)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
      images, masks = images.to(device), masks.to(device).float()

      outputs = model(images)
      loss = criterion(outputs, masks)

      total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
# TO DO
from torch import nn

# Define the loss and the optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 10  # epochs

# Run training
train_losses = []
val_losses = []

for epoch in range(num_epochs):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
  val_loss = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# TO DO
import random

model.eval() # put the model in the evaluation mode

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples form the data
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]

  with torch.no_grad():
    input_tensor = image.unsqueeze(0).to(device)
    output = model(input_tensor)
    pred = torch.sigmoid(output)
    pred = (pred > 0.5).float().cpu()

  # Display results
  axes[i, 0].imshow(image.permute(1, 2, 0).numpy())
  axes[i, 0].set_title("original Image")
  axes[i, 0].axis("off")

  axes[i, 1].imshow(mask.squeeze(), cmap="gray")
  axes[i, 1].set_title("Ground Truth Mask")
  axes[i, 1].axis("off")

  axes[i, 2].imshow(pred.squeeze(), cmap="gray")
  axes[i, 2].set_title("Prediction Mask")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()